# CountGD++ PrACo Official Adapter — LOCAL WSL2 (FULL)

**RTX 5060 8 GB / Blackwell sm_120 / Windows + WSL2 Ubuntu.**

This is a local adaptation of the verified Kaggle official-adapter notebook, not a new method.

- **SMOKE:** actual GPU inference on 3 image–query pairs; **no old prediction cache**, so it cannot pass by replaying old results.
- **FULL:** resumes 14,778 Kaggle predictions embedded in the notebook and processes remaining pairs.

**Precondition:** WSL2 + CUDA Toolkit >= 12.8, PyTorch cu128 supports sm_120. All data must be in `~/PrACo_Local/input/` (images extracted). Weights and synthetic exemplars download online; same resource SHA-256 as Kaggle required.

**8 GB risk:** do not proceed to FULL until SMOKE completes a real forward pass. If OOM occurs, don't reduce image size, thresholds or prompts for the paper without a separately identified experimental protocol.


In [ ]:
import os,sys,shutil,hashlib,json,time,zipfile,subprocess,random,gc,traceback
from pathlib import Path
import pandas as pd,numpy as np
from PIL import Image

MODE='FULL' # LOCAL WSL2, NO Kaggle mounts
FSC_DATASET_HINT='hihihi5'
RUN_BUDGET_HOURS=1000.0
STOP_RESERVE_MIN=25
SAVE_EVERY=29
ALLOW_COMMUNITY_HF_MIRROR=True  # third-party .pth with unverified equivalence; trust decision required
ALLOW_GOOGLE_DRIVE_SYNTH=True    # exact official synthetic exemplar archive from PrACo README
PRACO_COMMIT='438a968aef9d31ded7519733bd84e89baae68896'
PRACO_REPO='https://github.com/ciampluca/PrACo.git'
SYNTH_GDRIVE_ID='1XyE77D0bxEKDsPYvpaeD6KyXnSLye1Qg'
CKPT_HF_REPO='Bingjian/CountGD_plusplus'
CKPT_HF_FILE='countgd_plusplus.pth'

ROOT=Path.home()/'PrACo_Local'
INPUT=ROOT/'input'; WORK=ROOT/'work'
INPUT.mkdir(parents=True,exist_ok=True); WORK.mkdir(parents=True,exist_ok=True)
assert any(INPUT.rglob('*.jpg')), 'Extract FSC-147 images under ~/PrACo_Local/input/ first.'
OUT=WORK/('PrACo_OFFICIAL_ADAPTER_A_SMOKE' if MODE=='SMOKE' else 'PrACo_OFFICIAL_ADAPTER_A'); OUT.mkdir(parents=True,exist_ok=True)
R=WORK/'PrACo_native'
BASE=OUT/'official_baseline_pairs.csv'
PROTOCOL=OUT/'official_baseline_protocol.json'
SOURCE=OUT/'official_sources.json'
DEADLINE=time.monotonic()+RUN_BUDGET_HOURS*3600
safe=lambda:time.monotonic()<DEADLINE-STOP_RESERVE_MIN*60
print('MODE:',MODE,'| GPU check:',subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout[:200])
assert MODE in ('SMOKE','FULL')


In [ ]:
# SELF-CONTAINED RESUME: 14,778 completed OFFICIAL CountGD++ PrACo Test1 pairs.
# This is inference cache ONLY, not model weights. The exact same PrACo
# commit, model checkpoint SHA and synthetic-exemplar metadata are required.
# Keep stronger checkpoints from this Kaggle session; never overwrite them.
import base64, gzip, hashlib, pandas as pd
from pathlib import Path

_RESUME_EXPECTED_ROWS = 14778
_RESUME_BYTES_SHA256 = "76a7d2e54b7adee542b911024a74b44b9c7f9ce6442daaeb043ef11c8fec3006"
_RESUME_EXPECTED_PROTOCOL = "ab3a19afe6447047e71e4688fab39111b3c0ae90e02868a486bbecca341ba91c"
_RESUME_ENCODED = """..."""
_resume_bytes = gzip.decompress(base64.b64decode(_RESUME_ENCODED))
assert hashlib.sha256(_resume_bytes).hexdigest() == _RESUME_BYTES_SHA256, "Embedded cache corrupted"

_embedded = pd.read_csv(io.BytesIO(_resume_bytes)) if 'io' in globals() else pd.read_csv(
    __import__('io').BytesIO(_resume_bytes))
assert len(_embedded) == _RESUME_EXPECTED_ROWS
assert _embedded.pair_id.is_unique
assert set(_embedded.status) == {'ok'}
assert set(_embedded.protocol_sha256) == {_RESUME_EXPECTED_PROTOCOL}

_existing_n = 0
if BASE.exists():
    _old = pd.read_csv(BASE)
    _existing_n = len(_old)
    if _existing_n:
        assert _old.pair_id.is_unique, "Existing partial cache has duplicate pair IDs"
        assert set(_old.protocol_sha256) == {_RESUME_EXPECTED_PROTOCOL}, "Existing cache belongs to a different protocol"
if _existing_n < _RESUME_EXPECTED_ROWS:
    BASE.write_bytes(_resume_bytes)
    print(f"RESTORED {_RESUME_EXPECTED_ROWS:,} verified official-adapter predictions to {BASE}")
else:
    print(f"KEPT a newer existing cache: {_existing_n:,} rows (>= 14,778)")
del _RESUME_ENCODED, _resume_bytes


In [ ]:
# Clone exact official PrACo source and set path layout
if not R.is_dir():
    subprocess.run(['git','clone','-q',PRACO_REPO,str(R)],check=True)
subprocess.run(['git','-C',str(R),'checkout','-q',PRACO_COMMIT],check=True)
actual=subprocess.check_output(['git','-C',str(R),'rev-parse','HEAD'],text=True).strip()
assert actual==PRACO_COMMIT,actual
from pathlib import Path


In [ ]:
from pathlib import Path
HELPER_PATH=WORK/'praco_official_tools.py'
HELPER_PATH.write_text('"""Test-1 helpers: faithful negative-label roster, baseline audit and paired evaluation.\nBased on Ciampi et al. PrACo benchmark/benchmark.py and statistics_extractor.py.\nThis module only uses json, random, hashlib, pandas, numpy; no model weights/GT required for roster.\n"""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport random\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nBASE_COLS = ["pair_id","image_name","target","positive_class","exemplar_image","baseline_count","status"]\n\ndef digest_file(path, chunk=8*1024*1024):\n    h=hashlib.sha256()\n    with open(path,"rb") as f:\n        for b in iter(lambda:f.read(chunk),b""):\n            h.update(b)\n    return h.hexdigest()\n\ndef metadata(data_dir, split="test"):\n    d=Path(data_dir)\n    image_splits=json.loads((d/"Train_Test_Val_FSC_147.json").read_text())\n    class_splits=json.loads((d/"Split_Classes_FSC147.json").read_text())\n    classes=dict(line.rstrip("\\r\\n").split("\\t",1) for line in (d/"ImageClasses_FSC147.txt").read_text().splitlines() if "\\t" in line)\n    images=image_splits[split]\n    targets=class_splits[split]\n    assert len(images)==len(set(images)) and len(targets)==len(set(targets))\n    assert all(im in classes and classes[im] in targets for im in images)\n    return images,targets,classes,image_splits,class_splits\n\ndef official_schedule(images,targets,img_classes,seed=123):\n    """Reproduces PrACo Benchmark.run_negative_label_test random.choice for EVERY pair.\n    Important: includes the unused random draw on positive-class pairs, ensuring RNG alignment.\n    Candidate pool includes all FSC-147 split images, exactly as official img_classes.\n    """\n    buckets={c:[im for im,label in img_classes.items() if label==c] for c in targets}\n    assert all(buckets[c] for c in targets)\n    rng=random.Random(seed)\n    rows=[]\n    for im in images:\n        for target in targets:\n            chosen=rng.choice(buckets[target])\n            rows.append({"pair_id":f"{im}|||{target}","image_name":im,"target":target,\n                         "positive_class":img_classes[im],"exemplar_image":chosen})\n    df=pd.DataFrame(rows)\n    assert df.pair_id.is_unique\n    return df\n\ndef sha_roster(df):\n    return hashlib.sha256(df[["pair_id","image_name","target","positive_class","exemplar_image"]].to_csv(index=False).encode()).hexdigest()\n\ndef validate_baseline(df,roster,allow_partial=True):\n    required=set(BASE_COLS)\n    assert required.issubset(df.columns), f"Missing columns {required-set(df.columns)}"\n    assert df.pair_id.is_unique, "duplicate pair_id"\n    assert set(df.pair_id).issubset(set(roster.pair_id)),"row not in PrACo roster"\n    assert df.status.eq(\'ok\').all(),"non-ok status in baseline"\n    joined=df.merge(roster,on=\'pair_id\',suffixes=(\'\',\'_reference\'),validate=\'one_to_one\')\n    for k in [\'image_name\',\'target\',\'positive_class\',\'exemplar_image\']:\n        assert (joined[k].astype(str)==joined[k+\'_reference\'].astype(str)).all(),f"mismatched {k}"\n    nums=pd.to_numeric(df.baseline_count,errors=\'coerce\')\n    assert np.isfinite(nums).all() and (nums>=0).all(),"invalid counts"\n    if not allow_partial: assert len(df)==len(roster),f"Partial {len(df)}/{len(roster)}"\n    return True\n\ndef paired_test1(df,img_classes,gt_counts,expected_images=None,targets=None):\n    """PrACo Test1 metrics. PCCN uses strict smaller (<), per source code.\n    Requires exactly one positive-class count and all negative classes per image.\n    For class-mismatch, no claim that target was verified genuinely absent.\n    """\n    assert {\'image_name\',\'target\',\'baseline_count\',\'semantic_count\'}.issubset(df.columns)\n    assert not df[[\'image_name\',\'target\']].duplicated().any()\n    if expected_images is not None and targets is not None:\n        assert len(df)==len(expected_images)*len(targets)\n        assert set(df.image_name)==set(expected_images)\n        assert set(df.target)==set(targets)\n        assert df.groupby(\'image_name\').target.nunique().eq(len(targets)).all()\n    scores=[]\n    for col in [\'baseline_count\',\'semantic_count\']:\n        mat=df.pivot(index=\'image_name\',columns=\'target\',values=col)\n        if mat.isna().any().any():raise AssertionError(\'Incomplete predictions\')\n        rows=[]\n        for im in mat.index:\n            true_class=img_classes[im];gt=float(gt_counts[im]);pos=float(mat.at[im,true_class])\n            neg=float(mat.loc[im,mat.columns!=true_class].mean())\n            neg_nonzero=int((mat.loc[im,mat.columns!=true_class]>0).sum())\n            rows.append({\'image\':im,\'gt\':gt,\'pos\':pos,\'neg_mean\':neg,\'neg_nonzero\':neg_nonzero,\n                         \'neg_norm\':neg/gt,\'pos_norm\':pos/gt,\'positive_abs_err\':abs(pos-gt),\n                         \'negative_abs_gap\':abs(neg-gt)})\n        r=pd.DataFrame(rows)\n        scores.append({\'Model\':col,\'n_images\':len(r),\'AvgNP\':r.pos_norm.mean(),\n                       \'AvgNMN\':r.neg_norm.mean(),\n                       \'PCCN\':100*float((r.positive_abs_err<r.negative_abs_gap).mean()),\n                       \'MAE\':r.positive_abs_err.mean(),\n                       \'RMSE\':float(np.sqrt((r.positive_abs_err**2).mean())),\n                       \'mismatch_nonzero_rate\':r.neg_nonzero.sum()/(len(r)*(mat.shape[1]-1)),\n                       \'positive_zero\':int((r.pos==0).sum())})\n    return pd.DataFrame(scores)\n\ndef semantic_replay(baseline,first,second):\n    """Only two NO labels suppress; invalid/unknown replies are conservative."""\n    return 0.0 if float(baseline)>0 and first==\'NO\' and second==\'NO\' else float(baseline)\n',encoding='utf-8')
import sys
import importlib

sys.path.insert(0, str(WORK.resolve()))
importlib.invalidate_caches()
from praco_official_tools import metadata,official_schedule,sha_roster,digest_file,validate_baseline,paired_test1,semantic_replay
print('Helper loaded')


In [ ]:
# Build EXACT original test schedule before GT is read. All random exemplar choices
# are drawn for the full 1190 x 29 grid, even if SMOKE or resuming.
images,targets,img_classes,split_images,split_classes=metadata(R/'data')
assert len(images)==1190 and len(targets)==29
roster=official_schedule(images,targets,img_classes,seed=123)
assert len(roster)==34510
ROSTER_SHA=sha_roster(roster)
assert ROSTER_SHA == 'b73d78d0e21d089cdd875913fe861929904bc7842e600e84ff5a562d439eae39', 'PrACo roster changed; cannot reuse embedded cache'
roster.to_csv(OUT/'official_test1_roster.csv',index=False)
selected=roster.iloc[:3].copy() if MODE=='SMOKE' else roster
print('Official schedule:',len(roster),'| current mode',MODE,'pairs:',len(selected),'| sha:',ROSTER_SHA)

# Local input can be extracted into any directory beneath ~/PrACo_Local/input, OR a nested
# subdirectory. Do not assume its directory name.
# Locate all 1190 TEST image filenames recursively under LOCAL input.
print('LOCAL Input top-level:', [p.name for p in INPUT.iterdir()][:30])
image_names=set(images)
image_map={}
for p in INPUT.rglob('*'):
    if p.is_file() and p.suffix.lower() in ('.jpg','.jpeg') and p.name in image_names:
        image_map.setdefault(p.name,p)
missing=[im for im in images if im not in image_map]
if missing:
    print('Found',len(image_map),'of',len(images),'TEST images. Example found:',list(image_map.items())[:3])
    print('Expected images absent:',missing[:12])
    print('Searchable dataset dirs:',[str(p) for p in INPUT.rglob('FSC147_384_V2')][:8])
    raise FileNotFoundError(
        f'Missing {len(missing)}/{len(images)} TEST images. '
        'Check that local input contains the FSC147_384_V2 folder, and review extracted paths.'
    )
imgdir=R/'data'/'images_384_VarV2';imgdir.mkdir(parents=True,exist_ok=True)
for im,path in image_map.items():
    dest=imgdir/im
    if not dest.exists():dest.symlink_to(path.resolve())
print('Image links:',len(image_map),'| sample actual mounted path:',next(iter(image_map.values())))


In [ ]:
# SET UP PYTHON DEPENDENCIES, checkpoint and synthetic exemplars before GPU use.
import importlib.util
import torch
assert torch.cuda.is_available(),'WSL NVIDIA GPU not available in PyTorch; check driver and torch CUDA wheel.'
assert torch.cuda.get_device_capability(0) == (12,0), f'Expected RTX5060 Blackwell sm_120, got {torch.cuda.get_device_capability(0)}'
print('RTX 5060 VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
print('Torch:',torch.__version__,'| CUDA:',torch.version.cuda)

# Source pins are from the original author's PrACo adapter, excluding torch/torchvision
# because the RTX 5060 requires a modern sm_120-capable torch CUDA wheel.
model_root=R/'models'/'CountGDPlusPlus'
pkgs=['cython==3.0.9','submitit==1.5.1','termcolor==2.4.0','addict==2.4.0',
      'yapf==0.40.1','timm==0.9.16','transformers==4.39.1',
      'supervision==0.6.0','pycocotools','pyyaml','colorlog==6.8.2','inflect','gdown',
      'huggingface-hub','sentencepiece','ninja']
subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs],check=True)

ckpt_dir=model_root/'checkpoints';ckpt_dir.mkdir(exist_ok=True)
# Prefer an exact user-supplied checkpoint; else HF mirror, same source as prior pilots.
ckpt_hits=list(INPUT.rglob(CKPT_HF_FILE))
if ckpt_hits:
    CKPT=sorted(ckpt_hits)[0];CKPT_SOURCE='user Kaggle input (authenticity user supplied)'
else:
    assert ALLOW_COMMUNITY_HF_MIRROR,'No author checkpoint added and mirror disabled.'
    from huggingface_hub import hf_hub_download
    CKPT=Path(hf_hub_download(repo_id=CKPT_HF_REPO,filename=CKPT_HF_FILE))
    CKPT_SOURCE='third-party HF mirror (official equivalence NOT verified)'
assert CKPT.is_file() and CKPT.stat().st_size>1_000_000_000, 'Incomplete checkpoint'
ckpt_link=ckpt_dir/CKPT_HF_FILE
if not ckpt_link.exists():ckpt_link.symlink_to(CKPT.resolve())
CKPT_SHA=digest_file(CKPT)
assert CKPT_SHA == 'b89395d79711c9b9f13f3ef80cd2e528386fb1d70453f99443be4eccab33592f', 'CountGD++ weights differ from cached run; do not reuse cache'
print('Checkpoint',CKPT_SOURCE,'SHA256',CKPT_SHA)

# Synthetic exemplar archive: use data mounted by user, or download exact official ID.
synth_dest=model_root/'synthetic_exemplars';json_name='generated_exemplars.json'
def find_synth():
    # User may have mounted an extracted directory.
    for d in [synth_dest,*INPUT.rglob('synthetic_exemplars')]:
        if d.is_dir() and (d/json_name).is_file():return d
    return None
src=find_synth()
if src is None:
    # Search ZIP archives containing the expected JSON (not arbitrary unrelated zips).
    candidates=[]
    for z in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(z) as f:
                if any(name.endswith('/'+json_name) or name==json_name for name in f.namelist()):
                    candidates.append(z)
        except zipfile.BadZipFile:pass
    if not candidates:
        if not ALLOW_GOOGLE_DRIVE_SYNTH:raise FileNotFoundError('Add official synthetic_exemplars ZIP to Kaggle Input')
        import gdown
        download=WORK/'praco_official_synthetic_exemplars.zip'
        if not download.exists():
            print('Downloading official PrACo synthetic exemplars from Drive ID:',SYNTH_GDRIVE_ID)
            result=gdown.download(id=SYNTH_GDRIVE_ID,output=str(download),quiet=False)
            assert result and download.exists(), ('Google Drive quota/access error. Download author exemplar ZIP in browser '
                'and Add it as Kaggle Input; see HUONG_DAN.md.')
        candidates=[download]
    assert len(candidates)==1, f'Ambiguous exemplar zips: {candidates}'
    with zipfile.ZipFile(candidates[0]) as z:
        # Guard against zip slip (no traversal), zip bomb via decompressed size limit 10GB.
        assert sum(v.file_size for v in z.infolist())<10_000_000_000
        root_extract=WORK/'praco_synth_extracted';root_extract.mkdir(exist_ok=True)
        for v in z.infolist():
            rel=Path(v.filename)
            if rel.is_absolute() or '..' in rel.parts:raise ValueError('Unsafe ZIP path')
        z.extractall(root_extract)
    j=list(root_extract.rglob(json_name))
    assert len(j)==1,f'Exemplar ZIP must contain exactly one {json_name}, got {len(j)}'
    src=j[0].parent
assert (src/json_name).is_file()
if not synth_dest.exists():synth_dest.symlink_to(src.resolve())
meta_ex=json.loads((synth_dest/json_name).read_text())
needed=set(roster.image_name)|set(roster.exemplar_image)
missing_meta=needed-set(meta_ex)
missing_imgs=[im for im in needed if not (synth_dest/im).is_file()]
assert not missing_meta and not missing_imgs, (
    f'Exemplars incomplete: JSON missing {len(missing_meta)} (e.g {list(missing_meta)[:3]}); '
    f'JPG missing {len(missing_imgs)} (e.g {missing_imgs[:3]})')
SYNTH_SHA=digest_file(synth_dest/json_name)
assert SYNTH_SHA == '36f2142946464917f95b4503a7baa76a5d58a3a501a9db28554d9a64f0ec7fc2', 'Synthetic exemplars JSON differs from cached run; do not reuse cache'
print('Synthetic exemplars VALID:',len(needed),'images; JSON SHA:',SYNTH_SHA)

# Download BERT assets exactly as adapter expects.
bert=ckpt_dir/'bert-base-uncased'
if not (bert/'config.json').exists():
    subprocess.run([sys.executable,'download_bert.py'],cwd=str(model_root),check=True)
assert (bert/'config.json').exists()

# CUDA extension (native adapter). Fail with helpful log instead of silently changing model.
ops=model_root/'models'/'GroundingDINO'/'ops'
assert (ops/'setup.py').is_file(), 'PrACo adapter directory changed; check frozen SHA.'
log=OUT/'cuda_ops_build.log'

# Kaggle PyTorch (2026) removed implicit conversion of DeprecatedTypeProperties
# inside AT_DISPATCH_FLOATING_TYPES. The pinned PrACo CountGD++ native source
# uses value.type() at exactly two dispatch sites (forward/backward).
# This is a SOURCE-ONLY CUDA compatibility fix: no change to mathematical ops,
# model thresholds, weights, inference loop, or PrACo roster. Keep provenance.
cuda_src=ops/'src'/'cuda'/'ms_deform_attn_cuda.cu'
assert cuda_src.is_file(), f'Missing CUDA source: {cuda_src}'
legacy='AT_DISPATCH_FLOATING_TYPES(value.type(),'
modern='AT_DISPATCH_FLOATING_TYPES(value.scalar_type(),'
body=cuda_src.read_text()
if body.count(legacy)==2:
    cuda_src.write_text(body.replace(legacy,modern))
    print('Applied PyTorch AT_DISPATCH CUDA API compatibility patch (2 sites).')
elif body.count(modern)==2 and body.count(legacy)==0:
    print('CUDA API compatibility patch already applied.')
else:
    raise RuntimeError('Unexpected native CUDA source; aborting instead of blindly patching. '
                       f'Old occurrences={body.count(legacy)}, new={body.count(modern)}')
assert cuda_src.read_text().count(modern)==2
CUDA_PATCH_SHA=digest_file(cuda_src)
print('CUDA file SHA256 after compatibility patch:',CUDA_PATCH_SHA)

if importlib.util.find_spec('MultiScaleDeformableAttention') is None:
    import shutil as _shutil
    _nvcc=_shutil.which('nvcc')
    assert _nvcc, 'nvcc not found in WSL. Install CUDA Toolkit >=12.8 (toolkit only; no Linux NVIDIA display driver).'
    print('nvcc:',_nvcc)
    # Failed prior builds can leave partial object files; clean only local build.
    _shutil.rmtree(ops/'build',ignore_errors=True)
    env=os.environ.copy();env['MAX_JOBS']='1';env['TORCH_CUDA_ARCH_LIST']='12.0'
    nvcc_ver=subprocess.check_output([_nvcc,'--version'],text=True)
    print('NVCC:',nvcc_ver.splitlines()[-1]); print('Building for TORCH_CUDA_ARCH_LIST=12.0')
    with open(log,'w') as f:
        result=subprocess.run([sys.executable,'setup.py','build','install'],cwd=str(ops),
                              stdout=f,stderr=subprocess.STDOUT,env=env)
    if result.returncode:
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-100:])
        print('=== CUDA BUILD LOG: LAST 100 LINES ===\n'+tail)
        diag=WORK/'CS331_PrACo_OFFICIAL_ADAPTER_CUDA_BUILD_DIAGNOSTICS.zip'
        with zipfile.ZipFile(diag,'w',zipfile.ZIP_DEFLATED) as z:
            z.write(log,log.name)
            z.writestr('environment.json',json.dumps({
                'python':sys.version,'torch':torch.__version__,
                'torch_cuda':torch.version.cuda,'nvcc':_nvcc,
                'praco_commit':PRACO_COMMIT,'cuda_file_sha256':CUDA_PATCH_SHA,
                'patch':'AT_DISPATCH_FLOATING_TYPES(value.type() -> value.scalar_type() (2 sites)'
            },indent=2))
        print('Diagnostics ZIP:',diag)
        raise RuntimeError(f'CUDA extension compile failed after API patch. '
                           f'Send {diag} or log tail above. Code={result.returncode}')
assert importlib.util.find_spec('MultiScaleDeformableAttention') is not None

source={'adapter_repo':PRACO_REPO,'adapter_git_sha':PRACO_COMMIT,
        'checkpoint_source':CKPT_SOURCE,'checkpoint_sha256':CKPT_SHA,'checkpoint_size_bytes':CKPT.stat().st_size,
        'synthetic_exemplar_google_drive_id':SYNTH_GDRIVE_ID,
        'synthetic_json_sha256':SYNTH_SHA,'exemplar_required_images':len(needed),
        'official_schedule_sha256':ROSTER_SHA,'mode':MODE,'torch':torch.__version__,
        'cuda_compat_patch':'AT_DISPATCH_FLOATING_TYPES(value.type() -> value.scalar_type() in forward/backward',
        'cuda_source_patched_sha256':CUDA_PATCH_SHA}
SOURCE.write_text(json.dumps(source,indent=2))
print('FULL RESOURCE PREFLIGHT PASSED')


In [ ]:

# CELL 7 - FIX PACKAGE COLLISION + LOAD COUNTGD++

import os
import sys
import types
import importlib
import torch
from pathlib import Path
from importlib.machinery import ModuleSpec

R = Path.home() / "PrACo_Local/work/PrACo_native"
OUTER_MODELS = R / "models"
COUNTGD_ROOT = OUTER_MODELS / "CountGDPlusPlus"

assert (OUTER_MODELS / "loader.py").is_file()
assert (COUNTGD_ROOT / "groundingdino" / "util").is_dir()

# 1. Loai bo package models bi import nham
for name in list(sys.modules):
    if name == "models" or name.startswith("models."):
        del sys.modules[name]

# 2. Tao namespace models tro dung thu muc PrACo
pkg = types.ModuleType("models")
pkg.__path__ = [str(OUTER_MODELS)]
pkg.__package__ = "models"

spec = ModuleSpec(
    "models",
    loader=None,
    is_package=True,
)
spec.submodule_search_locations = [str(OUTER_MODELS)]
pkg.__spec__ = spec

sys.modules["models"] = pkg

# 3. Bo sung duong dan cho import tuyet doi
for p in [COUNTGD_ROOT, OUTER_MODELS, R]:
    path = str(p.resolve())
    while path in sys.path:
        sys.path.remove(path)
    sys.path.insert(0, path)

importlib.invalidate_caches()
os.chdir(R)

print("models paths:", list(pkg.__path__))

# 4. Import dung PrACo loader
from models.loader import load_model
print("PrACo loader: PASS")

# 5. Import GroundingDINO
import groundingdino.util

from models.CountGDPlusPlus.models.registry import (
    MODULE_BUILD_FUNCS,
)
from models.CountGDPlusPlus.models.GroundingDINO.groundingdino import (
    build_groundingdino,
)

if "groundingdino" not in MODULE_BUILD_FUNCS._module_dict:
    MODULE_BUILD_FUNCS.register(
        build_groundingdino,
        module_name="groundingdino",
    )

print("GroundingDINO registry: PASS")

# 6. Load CountGD++ official adapter
original_load = torch.load

def compatible_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return original_load(*args, **kwargs)

try:
    torch.load = compatible_load

    adapter = load_model(
        "CountGDPlusPlus",
        str(imgdir),
        split_images,
        split_classes,
        load_filtered_checkpoints=False,
        device="cuda:0",
        split="test",
    )
finally:
    torch.load = original_load

print("Official PrACo adapter loaded:", adapter.model_name)
print("MODEL LOAD: PASS")


In [ ]:
# Exact original inference loop from PrACo benchmark/benchmark.py, but with robust
# per-(image,target) resume/checkpoint and exact precomputed RNG exemplar schedule.
import pandas as pd,numpy as np
protocol={'type':'PrACo Test1 native CountGDPlusPlusModel','praco_commit':PRACO_COMMIT,
          'roster_sha256':ROSTER_SHA,'ckpt_sha256':CKPT_SHA,'synth_json_sha256':SYNTH_SHA,
          'seed':123,'negative_text':'original class positive/negative split',
          'text_method':'original get_text_prompt',
          'synth_and_pseudo_exemplars':True}
PROTOCOL_SHA=hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest()
PROTOCOL.write_text(json.dumps({**protocol,'protocol_sha256':PROTOCOL_SHA},indent=2))

known={}
# Resume only from matching OFFICIAL source/protocol; no TEXT_ONLY caches are accepted.
candidates=[BASE]
for zp in INPUT.rglob('*OFFICIAL_ADAPTER*PARTIAL*.zip'):
    try:
        with zipfile.ZipFile(zp) as z:
            if 'official_baseline_pairs.csv' in z.namelist():
                temp=OUT/('restore_'+zp.stem+'.csv')
                temp.write_bytes(z.read('official_baseline_pairs.csv'));candidates.append(temp)
    except zipfile.BadZipFile:continue
for f in candidates:
    if not f.exists():continue
    prior=pd.read_csv(f,keep_default_na=False)
    if prior.empty:continue
    assert set(prior.protocol_sha256)=={PROTOCOL_SHA},'Resume source differs from official protocol!'
    validate_baseline(prior,roster)
    for t in prior.to_dict('records'):
        if t['pair_id'] in known:
            assert np.isclose(float(known[t['pair_id']]['baseline_count']),float(t['baseline_count']))
        else:known[t['pair_id']]=t
print('Reused native adapter pairs:',len(known))

def atomic_frame(rows,path):
    if not rows:return
    temp=Path(str(path)+'.tmp');pd.DataFrame(list(rows.values())).sort_values('pair_id').to_csv(temp,index=False)
    temp.replace(path)

def export(stage):
    atomic_frame(known,BASE)
    state={'stage':stage,'mode':MODE,'completed':len(known),
           'expected':len(selected),'full_expected':len(roster),'protocol_sha256':PROTOCOL_SHA}
    (OUT/'run_state.json').write_text(json.dumps(state,indent=2))
    dest=WORK/f'CS331_PrACo_OFFICIAL_ADAPTER_A_{MODE}_{stage}.zip'
    with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED) as z:
        for p in [BASE,PROTOCOL,SOURCE,OUT/'run_state.json',OUT/'official_test1_roster.csv',OUT/'cuda_ops_build.log']:
            if p.exists():z.write(p,p.name)
    print('ARCHIVE:',dest,'rows:',len(known));return dest

for no,r in enumerate(selected.itertuples(index=False),1):
    if r.pair_id in known:continue
    if not safe():print('Time budget reached; exporting checkpoint');break
    with Image.open(imgdir/r.image_name) as f:photo=f.convert('RGB')
    pos_img=Image.open(synth_dest/r.image_name).convert('RGB')
    exemplar_img=Image.open(synth_dest/r.exemplar_image).convert('RGB')
    chosen=meta_ex[r.exemplar_image]; pos=meta_ex[r.image_name]
    positive_prompt=adapter.get_text_prompt(r.positive_class,r.image_name)
    start=time.time()
    try:
        with torch.inference_mode():
            if r.target==r.positive_class:
                count,_=adapter.infer(photo,positive_prompt,None,pos_img,None,[pos],None)
            else:
                negative_prompt=adapter.get_text_prompt(r.target,r.exemplar_image)
                count,_=adapter.infer(photo,negative_prompt,positive_prompt,
                                      exemplar_img,pos_img,[chosen],[pos])
    except torch.cuda.OutOfMemoryError as e:
        print('CUDA OOM on 8GB RTX5060, image=',r.image_name,'target=',r.target)
        export('PARTIAL')
        raise RuntimeError('Insufficient VRAM at ORIGINAL PrACo 800px inference. No predictions were fabricated. Stop, do not change protocol silently.') from e
    assert np.isfinite(float(count)) and float(count)>=0
    known[r.pair_id]={'pair_id':r.pair_id,'image_name':r.image_name,'target':r.target,
           'positive_class':r.positive_class,'exemplar_image':r.exemplar_image,
           'baseline_count':float(count),'status':'ok',
           'elapsed_sec':round(time.time()-start,3),'protocol_sha256':PROTOCOL_SHA}
    if no%SAVE_EVERY==0:
        export('PARTIAL')
        print('PROGRESS:',len(known),'/',len(selected),'elapsed after this pair:',round(time.time()-start,2))

complete_all=MODE=='FULL' and len(known)==len(roster)
if complete_all:
    all_records=pd.DataFrame(known.values())
    validate_baseline(all_records,roster,allow_partial=False)
    matrix=all_records.pivot(index='image_name',columns='target',values='baseline_count')
    matrix=matrix.reindex(index=images,columns=targets)
    assert not matrix.isna().any().any()
    matrix.to_csv(OUT/'Inference_Test1_CountGDPlusPlus_test.csv')
    print('FULL official Test1 baseline matrix sealed:',matrix.shape)
final=export('FULL' if complete_all else 'PARTIAL')
if complete_all:
    with zipfile.ZipFile(final,'a',zipfile.ZIP_DEFLATED) as z:
        z.write(OUT/'Inference_Test1_CountGDPlusPlus_test.csv','Inference_Test1_CountGDPlusPlus_test.csv')
from IPython.display import FileLink,display
display(FileLink(str(final)))
print('COMPLETE?',complete_all)
